# Seminar 5: Incremental ALS on VK-LSVD

## Goals

In this seminar we will:
1. Work with the **VK-LSVD** large-scale short-video recommendation dataset
2. Use the `implicit` library for ALS with **native incremental training** support
3. Implement **implicit ALS (iALS)** from scratch with warm-start capability
4. Construct various target signals: raw, derived (**watch_ratio**), and **composite** weighted targets
5. Compare **cold-start vs warm-start** training week by week
6. Evaluate which target signals produce the best recommendations

In [ ]:
!pip install implicit polars huggingface_hub

In [ ]:
import os
import time
import copy
from pathlib import Path
from collections import defaultdict

import numpy as np
from scipy import sparse
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

plt.style.use("ggplot")

## 2. VK-LSVD Dataset

**VK-LSVD** is the largest open industrial short-video recommendation dataset:

- **40B** unique user–item interactions with rich feedback (`timespent`, `like`, `dislike`, `share`, `bookmark`, `click_on_author`, `open_comments`) and context (`place`, `platform`, `agent`)
- **10M** users with demographic metadata (age, gender, geo)
- **20M** short videos with duration, author, and content embeddings
- Data organized in **weekly** parquet files with **global temporal ordering** across six consecutive months

We use the **`up0.001_ip0.001`** subsample (~10K most active users, ~20K most popular items) for manageable runtime.

Dataset: https://huggingface.co/datasets/deepvk/VK-LSVD

In [ ]:
SUBSAMPLE = "up0.001_ip0.001"
DATA_DIR = Path("VK-LSVD")

train_week_files = [
    f"subsamples/{SUBSAMPLE}/train/week_{i:02d}.parquet" for i in range(25)
]
val_week_files = [f"subsamples/{SUBSAMPLE}/validation/week_25.parquet"]
metadata_files = [
    "metadata/users_metadata.parquet",
    "metadata/items_metadata.parquet",
]

for f in tqdm(train_week_files + val_week_files + metadata_files, desc="Downloading"):
    hf_hub_download(
        repo_id="deepvk/VK-LSVD",
        repo_type="dataset",
        filename=f,
        local_dir=str(DATA_DIR),
    )
print("Download complete.")

In [ ]:
train_weeks_dfs = []
for i in tqdm(range(25), desc="Loading train weeks"):
    path = DATA_DIR / f"subsamples/{SUBSAMPLE}/train/week_{i:02d}.parquet"
    wdf = pl.read_parquet(path)
    wdf = wdf.with_columns(pl.lit(i).cast(pl.UInt8).alias("week"))
    train_weeks_dfs.append(wdf)

train_df = pl.concat(train_weeks_dfs)
val_df = pl.read_parquet(
    DATA_DIR / f"subsamples/{SUBSAMPLE}/validation/week_25.parquet"
)

print(f"Train interactions: {len(train_df):,}")
print(f"Val interactions:   {len(val_df):,}")
print(f"Train users:        {train_df['user_id'].n_unique():,}")
print(f"Train items:        {train_df['item_id'].n_unique():,}")
train_df.head()

In [ ]:
users_meta = pl.read_parquet(DATA_DIR / "metadata/users_metadata.parquet")
items_meta = pl.read_parquet(DATA_DIR / "metadata/items_metadata.parquet")

train_user_set = set(train_df["user_id"].unique().to_list())
train_item_set = set(train_df["item_id"].unique().to_list())
users_meta = users_meta.filter(pl.col("user_id").is_in(train_user_set))
items_meta = items_meta.filter(pl.col("item_id").is_in(train_item_set))

print(f"Users metadata: {len(users_meta):,} rows")
print(f"Items metadata: {len(items_meta):,} rows")
print(f"Item duration range: {items_meta['duration'].min()} – {items_meta['duration'].max()} seconds")
items_meta.head()

In [ ]:
print("=== Target signal statistics (train) ===")
print(f"timespent — mean: {train_df['timespent'].mean():.1f}, "
      f"median: {train_df['timespent'].median():.1f}")
for col in ["like", "dislike", "share", "bookmark",
            "click_on_author", "open_comments"]:
    rate = train_df[col].sum() / len(train_df) * 100
    print(f"{col:20s} — rate: {rate:.3f}%  (count: {train_df[col].sum():,})")

weekly_counts = (
    train_df.group_by("week")
    .agg(pl.len().alias("n_interactions"))
    .sort("week")
)
print("\n=== Interactions per week ===")
print(weekly_counts)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ts = train_df["timespent"].to_numpy()
axes[0, 0].hist(ts[ts > 0], bins=50, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Timespent (seconds)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Timespent distribution (timespent > 0)")
axes[0, 0].set_yscale("log")

wc = weekly_counts.to_pandas()
axes[0, 1].bar(wc["week"], wc["n_interactions"], edgecolor="black", alpha=0.7)
axes[0, 1].set_xlabel("Week")
axes[0, 1].set_ylabel("Interactions")
axes[0, 1].set_title("Interactions per week")

signals = ["like", "share", "bookmark", "click_on_author",
           "open_comments", "dislike"]
rates = [train_df[s].sum() / len(train_df) * 100 for s in signals]
axes[1, 0].barh(signals, rates, edgecolor="black", alpha=0.7)
axes[1, 0].set_xlabel("Rate (%)")
axes[1, 0].set_title("Binary signal rates")

ipu = train_df.group_by("user_id").agg(pl.len().alias("cnt"))["cnt"].to_numpy()
axes[1, 1].hist(ipu, bins=50, edgecolor="black", alpha=0.7)
axes[1, 1].set_xlabel("Items per user")
axes[1, 1].set_ylabel("Number of users")
axes[1, 1].set_title("Items per user distribution")

plt.tight_layout()
plt.show()

## 3. Theory

### 3.1 Explicit ALS (recap)

For a user–item rating matrix $R \in \mathbb{R}^{m \times n}$ with observed entries $\Omega$, explicit ALS minimizes:

$$
\min_{U, V} \sum_{(u,i) \in \Omega} (r_{ui} - u_u^\top v_i)^2 + \lambda \left(\|U\|_F^2 + \|V\|_F^2\right)
$$

Fixing $V$, the closed-form update for user $u$ is:

$$
u_u = (V_u^\top V_u + \lambda I)^{-1} V_u^\top r_u
$$

where $V_u$ contains rows of $V$ for items rated by user $u$, and $r_u$ is the vector of those ratings.

See **Seminar 4** for the full derivation.

### 3.2 Implicit ALS — iALS (Hu, Koren, Volinsky, 2008)

For implicit feedback (clicks, watches, likes) we don't have explicit ratings. Instead, we observe **interactions** $r_{ui} \geq 0$ and define:

**Preference** (did the user like the item?):

$$
p_{ui} = \begin{cases} 1, & r_{ui} > 0 \\ 0, & r_{ui} = 0 \end{cases}
$$

**Confidence** (how sure are we?):

$$
c_{ui} = 1 + \alpha \, r_{ui}
$$

Higher $r_{ui}$ (e.g., longer watch time) $\Rightarrow$ higher confidence that the user truly prefers the item. Unobserved items have minimal confidence $c_{ui} = 1$ that $p_{ui} = 0$.

#### Objective

Unlike explicit ALS which sums over observed entries only, iALS optimizes over **all** user–item pairs:

$$
\min_{U, V} \sum_u \sum_i c_{ui} \left(p_{ui} - u_u^\top v_i\right)^2 + \lambda \left(\|U\|_F^2 + \|V\|_F^2\right)
$$

#### User factor update

Fix $V$, take the gradient w.r.t. $u_u$ and set to zero:

$$
u_u = \left(V^\top C^u V + \lambda I\right)^{-1} V^\top C^u p_u
$$

where $C^u = \mathrm{diag}(c_{u1}, \ldots, c_{un})$.

#### Efficient computation

Forming $V^\top C^u V$ naively costs $O(n k^2)$ — too expensive. The key trick:

$$
V^\top C^u V = V^\top V + V^\top (C^u - I) V
$$

Since $C^u - I$ is non-zero only for **observed** items (where $c_{ui} - 1 = \alpha\, r_{ui}$), the second term involves only the small set $I_u$ of items user $u$ interacted with:

$$
V^\top C^u V = \underbrace{V^\top V}_{\text{precompute once}} + V_{I_u}^\top \, \mathrm{diag}(\alpha\, r_u) \, V_{I_u}
$$

Similarly, $V^\top C^u p_u = V_{I_u}^\top c_u$ since $p_{ui} = 0$ for unobserved items.

The final update:

$$
\boxed{u_u = \left(V^\top V + V_{I_u}^\top \, \mathrm{diag}(\alpha\, r_u) \, V_{I_u} + \lambda I\right)^{-1} V_{I_u}^\top c_u}
$$

where $c_u = 1 + \alpha\, r_u$ for items in $I_u$.

#### Item factor update

By symmetry, fixing $U$:

$$
v_i = \left(U^\top U + U_{U_i}^\top \, \mathrm{diag}(\alpha\, r_i) \, U_{U_i} + \lambda I\right)^{-1} U_{U_i}^\top c_i
$$

### 3.3 Incremental (Warm-Start) Training

In production, new interactions arrive continuously. **Cold-start** training re-initializes factors randomly each time — wasteful when factors from the previous period are already good.

**Warm-start** approach:
1. Train on data from period $t$, obtain factors $U^{(t)}, V^{(t)}$
2. When period $t+1$ data arrives, initialize with $U^{(t)}, V^{(t)}$ instead of random
3. Run a few ALS iterations to refine

**Benefits:**
- **Faster convergence**: starting near the optimum requires fewer iterations
- **Temporal continuity**: factors evolve smoothly
- **New entities**: users/items appearing for the first time get random initialization; existing ones keep their learned factors

## 4. ALS via `implicit` Library

The [`implicit`](https://github.com/benfred/implicit) library provides a highly optimized C++ implementation of iALS with:

- `fit(user_items)` — full training from scratch
- `partial_fit_users(userids, user_items)` / `partial_fit_items(itemids, item_users)` — **native incremental updates**
- `recommend()` — fast top-N recommendation with item filtering

We use this as our **library baseline**.

In [ ]:
all_user_ids = np.sort(train_df["user_id"].unique().to_numpy())
all_item_ids = np.sort(train_df["item_id"].unique().to_numpy())
n_users = len(all_user_ids)
n_items = len(all_item_ids)

print(f"n_users={n_users:,}, n_items={n_items:,}")


def map_ids(ids, sorted_all):
    """Map raw IDs to contiguous indices via binary search."""
    return np.searchsorted(sorted_all, ids)


user_idx_all = map_ids(train_df["user_id"].to_numpy(), all_user_ids)
item_idx_all = map_ids(train_df["item_id"].to_numpy(), all_item_ids)

R_train_any = sparse.csr_matrix(
    (np.ones(len(train_df), dtype=np.float32),
     (user_idx_all, item_idx_all)),
    shape=(n_users, n_items),
)

ts_vals = train_df["timespent"].to_numpy().astype(np.float32)
ts_mask = ts_vals > 0
R_timespent = sparse.csr_matrix(
    (ts_vals[ts_mask], (user_idx_all[ts_mask], item_idx_all[ts_mask])),
    shape=(n_users, n_items),
)

print(f"R_train_any: {R_train_any.shape}, nnz={R_train_any.nnz:,}")
print(f"R_timespent: {R_timespent.shape}, nnz={R_timespent.nnz:,}")

In [ ]:
from implicit.als import AlternatingLeastSquares

N_FACTORS = 32
REG = 0.01
ALPHA = 1.0
N_ITERS = 15

lib_als = AlternatingLeastSquares(
    factors=N_FACTORS,
    regularization=REG,
    alpha=ALPHA,
    iterations=N_ITERS,
    random_state=42,
)

t0 = time.time()
lib_als.fit(R_timespent, show_progress=True)
print(f"Library ALS fit in {time.time() - t0:.1f}s")
print(f"User factors: {lib_als.user_factors.shape}")
print(f"Item factors: {lib_als.item_factors.shape}")

## 5. Implicit ALS from Scratch

We implement iALS following the derivation in Section 3.2, with two training modes:

- `fit(R)` — full training with random initialization
- `partial_fit(R)` — warm-start training that reuses previously learned factors

$$
V^\top C^u V = \underbrace{V^\top V}_{\text{precompute once}} + V_{I_u}^\top \, \mathrm{diag}(\alpha\, r_u) \, V_{I_u}
$$

In [ ]:
class ImplicitALS:
    def __init__(self, n_factors=32, n_iters=15, reg=0.01, alpha=1.0,
                 random_state=42):
        self.n_factors = n_factors
        self.n_iters = n_iters
        self.reg = reg
        self.alpha = alpha
        self.random_state = random_state
        self.user_factors = None
        self.item_factors = None

    def _init_factors(self, n_users, n_items):
        rng = np.random.default_rng(self.random_state)
        self.user_factors = (
            rng.standard_normal((n_users, self.n_factors)) * 0.01
        )
        self.item_factors = (
            rng.standard_normal((n_items, self.n_factors)) * 0.01
        )

    def _als_step(self, R_csr, R_csc):
        n_users, n_items = R_csr.shape
        k = self.n_factors
        reg_I = self.reg * np.eye(k)

        # --- update users ---
        VtV = self.item_factors.T @ self.item_factors
        for u in range(n_users):
            s, e = R_csr.indptr[u], R_csr.indptr[u + 1]
            if s == e:
                continue
            idx = R_csr.indices[s:e]
            r = R_csr.data[s:e].astype(np.float64)
            V_u = self.item_factors[idx]
            alpha_r = self.alpha * r
            A = VtV + (V_u.T * alpha_r) @ V_u + reg_I
            b = V_u.T @ (1.0 + alpha_r)
            self.user_factors[u] = np.linalg.solve(A, b)

        # --- update items ---
        UtU = self.user_factors.T @ self.user_factors
        for i in range(n_items):
            s, e = R_csc.indptr[i], R_csc.indptr[i + 1]
            if s == e:
                continue
            idx = R_csc.indices[s:e]
            r = R_csc.data[s:e].astype(np.float64)
            U_i = self.user_factors[idx]
            alpha_r = self.alpha * r
            A = UtU + (U_i.T * alpha_r) @ U_i + reg_I
            b = U_i.T @ (1.0 + alpha_r)
            self.item_factors[i] = np.linalg.solve(A, b)

    def fit(self, R_csr):
        """Full training with random initialization."""
        n_users, n_items = R_csr.shape
        self._init_factors(n_users, n_items)
        R_csc = R_csr.tocsc()
        for it in tqdm(range(self.n_iters), desc="iALS fit"):
            self._als_step(R_csr, R_csc)
        return self

    def partial_fit(self, R_csr, n_iters=None):
        """Warm-start training: reuse existing factors."""
        n_users, n_items = R_csr.shape
        if self.user_factors is None:
            return self.fit(R_csr)

        if n_users > self.user_factors.shape[0]:
            rng = np.random.default_rng(self.random_state + 100)
            extra = rng.standard_normal(
                (n_users - self.user_factors.shape[0], self.n_factors)
            ) * 0.01
            self.user_factors = np.vstack([self.user_factors, extra])
        if n_items > self.item_factors.shape[0]:
            rng = np.random.default_rng(self.random_state + 200)
            extra = rng.standard_normal(
                (n_items - self.item_factors.shape[0], self.n_factors)
            ) * 0.01
            self.item_factors = np.vstack([self.item_factors, extra])

        R_csc = R_csr.tocsc()
        iters = n_iters if n_iters is not None else self.n_iters
        for it in tqdm(range(iters), desc="iALS partial_fit"):
            self._als_step(R_csr, R_csc)
        return self

    def predict_for_user(self, user_idx):
        return self.item_factors @ self.user_factors[user_idx]

In [ ]:
n_u_small, n_i_small, k_small = 50, 80, 8
rng = np.random.default_rng(0)
rows_s = rng.integers(0, n_u_small, size=500)
cols_s = rng.integers(0, n_i_small, size=500)
vals_s = rng.random(500).astype(np.float32) * 5

R_small = sparse.csr_matrix(
    (vals_s, (rows_s, cols_s)), shape=(n_u_small, n_i_small)
)

ials_small = ImplicitALS(n_factors=k_small, n_iters=5, reg=0.01,
                         alpha=1.0, random_state=0)
ials_small.fit(R_small)

assert ials_small.user_factors.shape == (n_u_small, k_small)
assert ials_small.item_factors.shape == (n_i_small, k_small)

scores = ials_small.predict_for_user(0)
assert scores.shape == (n_i_small,)
assert np.isfinite(scores).all()

ials_small2 = ImplicitALS(n_factors=k_small, n_iters=5, reg=0.01,
                          alpha=1.0, random_state=0)
ials_small2.partial_fit(R_small, n_iters=3)
assert ials_small2.user_factors.shape == (n_u_small, k_small)

print("All sanity checks passed.")

## 6. Target Construction

We build multiple target signals from the interaction data:

### 6.1 Derived feature: watch_ratio

Join interactions with `items_metadata` to get video `duration`, then compute:

$$\text{watch\_ratio} = \min\!\left(\frac{\text{timespent}}{\text{duration}},\; 2.0\right)$$

Capped at 2.0 to handle replays/outliers. Interpretation:
- $< 1$: user skipped early
- $\approx 1$: watched the full video
- $> 1$: replayed

### 6.2 Simple targets

Individual signal CSR matrices: `timespent`, `watch_ratio`, `like`, `share`, `bookmark`, `click_on_author`, `open_comments`.

### 6.3 Composite weighted targets

Combine multiple signals into a single scalar per interaction:

- **Engagement-heavy**: $1.0 \cdot \text{watch\_ratio} + 0.5 \cdot \text{like} + 0.3 \cdot \text{open\_comments}$
- **Action-heavy**: $0.3 \cdot \text{watch\_ratio} + 1.0 \cdot \text{like} + 1.5 \cdot \text{share} + 1.0 \cdot \text{bookmark}$
- **Full composite**: $0.5 \cdot \text{watch\_ratio} + 1.0 \cdot \text{like} + 1.5 \cdot \text{share} + 1.0 \cdot \text{bookmark} + 0.3 \cdot \text{click\_on\_author} + 0.3 \cdot \text{open\_comments} - 0.5 \cdot \text{dislike}$

In [ ]:
train_with_dur = train_df.join(
    items_meta.select(["item_id", "duration"]), on="item_id", how="left"
)
train_with_dur = train_with_dur.with_columns(
    pl.when(pl.col("duration") > 0)
    .then(
        (pl.col("timespent").cast(pl.Float32)
         / pl.col("duration").cast(pl.Float32)).clip(0, 2.0)
    )
    .otherwise(0.0)
    .alias("watch_ratio")
)

wr_vals = train_with_dur["watch_ratio"].to_numpy().astype(np.float32)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(wr_vals[wr_vals > 0], bins=100, edgecolor="black", alpha=0.7)
ax.set_xlabel("watch_ratio")
ax.set_ylabel("Count")
ax.set_title("Watch ratio distribution (watch_ratio > 0)")
ax.set_yscale("log")
ax.axvline(x=1.0, color="red", linestyle="--", label="full watch")
ax.legend()
plt.tight_layout()
plt.show()


def build_csr(vals, user_idx, item_idx, shape, min_val=0.0):
    """Build CSR keeping only entries with value > min_val."""
    mask = vals > min_val
    return sparse.csr_matrix(
        (vals[mask], (user_idx[mask], item_idx[mask])), shape=shape
    )


shape = (n_users, n_items)

R_watch_ratio = build_csr(wr_vals, user_idx_all, item_idx_all, shape)

target_matrices = {"timespent": R_timespent, "watch_ratio": R_watch_ratio}

for col in ["like", "share", "bookmark", "click_on_author", "open_comments"]:
    bin_vals = train_df[col].to_numpy().astype(np.float32)
    target_matrices[col] = build_csr(
        bin_vals, user_idx_all, item_idx_all, shape
    )

for name, mat in target_matrices.items():
    print(f"{name:20s}: nnz={mat.nnz:>12,}  "
          f"density={mat.nnz / (n_users * n_items) * 100:.2f}%")

In [ ]:
bool_cols = {
    c: train_with_dur[c].to_numpy().astype(np.float32)
    for c in ["like", "dislike", "share", "bookmark",
              "click_on_author", "open_comments"]
}

engagement_vals = (
    1.0 * wr_vals
    + 0.5 * bool_cols["like"]
    + 0.3 * bool_cols["open_comments"]
)
target_matrices["composite: engagement"] = build_csr(
    engagement_vals, user_idx_all, item_idx_all, shape
)

action_vals = (
    0.3 * wr_vals
    + 1.0 * bool_cols["like"]
    + 1.5 * bool_cols["share"]
    + 1.0 * bool_cols["bookmark"]
)
target_matrices["composite: action"] = build_csr(
    action_vals, user_idx_all, item_idx_all, shape
)

full_vals = (
    0.5 * wr_vals
    + 1.0 * bool_cols["like"]
    + 1.5 * bool_cols["share"]
    + 1.0 * bool_cols["bookmark"]
    + 0.3 * bool_cols["click_on_author"]
    + 0.3 * bool_cols["open_comments"]
    - 0.5 * bool_cols["dislike"]
)
target_matrices["composite: full"] = build_csr(
    full_vals, user_idx_all, item_idx_all, shape
)

for name in ["composite: engagement", "composite: action", "composite: full"]:
    mat = target_matrices[name]
    print(f"{name:30s}: nnz={mat.nnz:>12,}  "
          f"density={mat.nnz / (n_users * n_items) * 100:.2f}%")

## 7. Evaluation

- **Global temporal split**: train on weeks 0–24, evaluate on week 25 (validation)
- Filter validation to users and items present in training
- Metrics: **HitRate@K**, **MRR@K**, **NDCG@K** (K = 10)

In [ ]:
def evaluate(user_factors, item_factors, R_filter, val_user_items, k=10):
    """Compute HitRate@K, MRR@K, NDCG@K.

    R_filter: CSR matrix used to filter out seen items during scoring.
    val_user_items: dict[user_idx -> list[item_idx]] ground truth.
    """
    hits = 0
    mrr_sum = 0.0
    ndcg_sum = 0.0
    n_eval = 0

    for u, true_items in val_user_items.items():
        scores = item_factors @ user_factors[u]

        s, e = R_filter.indptr[u], R_filter.indptr[u + 1]
        scores[R_filter.indices[s:e]] = -np.inf

        n_score = len(scores)
        if k >= n_score:
            top_k = np.argsort(-scores)
        else:
            top_k = np.argpartition(-scores, k)[:k]
            top_k = top_k[np.argsort(-scores[top_k])]

        true_set = set(true_items)

        hits += int(bool(true_set & set(top_k)))

        for rank, item in enumerate(top_k, 1):
            if item in true_set:
                mrr_sum += 1.0 / rank
                break

        dcg = sum(
            1.0 / np.log2(r + 2)
            for r, item in enumerate(top_k)
            if item in true_set
        )
        n_rel = min(k, len(true_items))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(n_rel))
        ndcg_sum += dcg / idcg if idcg > 0 else 0.0

        n_eval += 1

    if n_eval == 0:
        return {f"HitRate@{k}": 0, f"MRR@{k}": 0, f"NDCG@{k}": 0}
    return {
        f"HitRate@{k}": hits / n_eval,
        f"MRR@{k}": mrr_sum / n_eval,
        f"NDCG@{k}": ndcg_sum / n_eval,
    }


val_uid_np = val_df["user_id"].to_numpy()
val_iid_np = val_df["item_id"].to_numpy()
val_in_train = np.isin(val_uid_np, all_user_ids) & np.isin(val_iid_np, all_item_ids)

val_u_idx = map_ids(val_uid_np[val_in_train], all_user_ids)
val_i_idx = map_ids(val_iid_np[val_in_train], all_item_ids)

val_user_items = defaultdict(list)
for u, i in zip(val_u_idx, val_i_idx):
    val_user_items[u].append(i)
val_user_items = dict(val_user_items)

print(f"Validation users (in train): {len(val_user_items):,}")
print(f"Avg val items per user: "
      f"{np.mean([len(v) for v in val_user_items.values()]):.1f}")

In [ ]:
K = 10

metrics_lib = evaluate(
    lib_als.user_factors, lib_als.item_factors,
    R_train_any, val_user_items, k=K
)
print(f"Library ALS (timespent): {metrics_lib}")

ials_scratch = ImplicitALS(
    n_factors=N_FACTORS, n_iters=N_ITERS, reg=REG, alpha=ALPHA,
    random_state=42,
)
t0 = time.time()
ials_scratch.fit(R_timespent)
print(f"From-scratch iALS fit in {time.time() - t0:.1f}s")

metrics_scratch = evaluate(
    ials_scratch.user_factors, ials_scratch.item_factors,
    R_train_any, val_user_items, k=K
)
print(f"From-scratch iALS (timespent): {metrics_scratch}")

## 8. Incremental Training

We compare **cold start** (random initialization each time) vs **warm start** (reuse previous factors) as training data grows week by week.

- **Library ALS**: cold = `fit()`, warm = `partial_fit_users()` + `partial_fit_items()` alternating
- **From-scratch iALS**: cold = `fit()`, warm = `partial_fit()`

In [ ]:
EVAL_WEEKS = [0, 4, 9, 14, 19, 24]
WARM_ITERS = 5

def build_cumulative_csr(week_limit, target="timespent"):
    """Build CSR from weeks 0..week_limit (inclusive)."""
    mask_week = train_df["week"].to_numpy() <= week_limit
    u_idx = user_idx_all[mask_week]
    i_idx = item_idx_all[mask_week]
    if target == "timespent":
        vals = ts_vals[mask_week]
    elif target == "any":
        vals = np.ones(mask_week.sum(), dtype=np.float32)
    else:
        raise ValueError(target)
    pos = vals > 0
    return sparse.csr_matrix(
        (vals[pos], (u_idx[pos], i_idx[pos])),
        shape=(n_users, n_items),
    )


for w in EVAL_WEEKS:
    R_w = build_cumulative_csr(w)
    print(f"Weeks 0–{w:2d}: nnz={R_w.nnz:>10,}")

In [ ]:
cold_metrics, warm_metrics = {}, {}
cold_times, warm_times = {}, {}

model_warm_lib = None

for w in EVAL_WEEKS:
    R_cum = build_cumulative_csr(w)
    R_cum_filter = sparse.csr_matrix(
        (np.ones(R_cum.nnz, dtype=np.float32),
         R_cum.nonzero()),
        shape=(n_users, n_items),
    )

    # --- cold start (library) ---
    t0 = time.time()
    m_cold = AlternatingLeastSquares(
        factors=N_FACTORS, regularization=REG, alpha=ALPHA,
        iterations=N_ITERS, random_state=42,
    )
    m_cold.fit(R_cum, show_progress=False)
    cold_times[w] = time.time() - t0
    cold_metrics[w] = evaluate(
        m_cold.user_factors, m_cold.item_factors,
        R_cum_filter, val_user_items, k=K,
    )

    # --- warm start (library) ---
    t0 = time.time()
    if model_warm_lib is None:
        model_warm_lib = AlternatingLeastSquares(
            factors=N_FACTORS, regularization=REG, alpha=ALPHA,
            iterations=N_ITERS, random_state=42,
        )
        model_warm_lib.fit(R_cum, show_progress=False)
    else:
        R_cum_T = R_cum.T.tocsr()
        for _ in range(WARM_ITERS):
            model_warm_lib.partial_fit_users(
                np.arange(n_users), R_cum
            )
            model_warm_lib.partial_fit_items(
                np.arange(n_items), R_cum_T
            )
    warm_times[w] = time.time() - t0
    warm_metrics[w] = evaluate(
        model_warm_lib.user_factors, model_warm_lib.item_factors,
        R_cum_filter, val_user_items, k=K,
    )

    print(f"Week {w:2d} | cold {cold_times[w]:5.1f}s "
          f"{cold_metrics[w]} | warm {warm_times[w]:5.1f}s "
          f"{warm_metrics[w]}")

In [ ]:
cold_metrics_s, warm_metrics_s = {}, {}
cold_times_s, warm_times_s = {}, {}

model_warm_scratch = None

for w in EVAL_WEEKS:
    R_cum = build_cumulative_csr(w)
    R_cum_filter = sparse.csr_matrix(
        (np.ones(R_cum.nnz, dtype=np.float32),
         R_cum.nonzero()),
        shape=(n_users, n_items),
    )

    # --- cold start (from scratch) ---
    t0 = time.time()
    m_cold_s = ImplicitALS(
        n_factors=N_FACTORS, n_iters=N_ITERS, reg=REG, alpha=ALPHA,
        random_state=42,
    )
    m_cold_s.fit(R_cum)
    cold_times_s[w] = time.time() - t0
    cold_metrics_s[w] = evaluate(
        m_cold_s.user_factors, m_cold_s.item_factors,
        R_cum_filter, val_user_items, k=K,
    )

    # --- warm start (from scratch) ---
    t0 = time.time()
    if model_warm_scratch is None:
        model_warm_scratch = ImplicitALS(
            n_factors=N_FACTORS, n_iters=N_ITERS, reg=REG, alpha=ALPHA,
            random_state=42,
        )
        model_warm_scratch.fit(R_cum)
    else:
        model_warm_scratch.partial_fit(R_cum, n_iters=WARM_ITERS)
    warm_times_s[w] = time.time() - t0
    warm_metrics_s[w] = evaluate(
        model_warm_scratch.user_factors, model_warm_scratch.item_factors,
        R_cum_filter, val_user_items, k=K,
    )

    print(f"Week {w:2d} | cold {cold_times_s[w]:6.1f}s "
          f"{cold_metrics_s[w]} | warm {warm_times_s[w]:6.1f}s "
          f"{warm_metrics_s[w]}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metric_key = f"MRR@{K}"

# Library ALS: metrics
weeks = sorted(cold_metrics.keys())
axes[0, 0].plot(weeks, [cold_metrics[w][metric_key] for w in weeks],
                "o-", label="Cold start")
axes[0, 0].plot(weeks, [warm_metrics[w][metric_key] for w in weeks],
                "s--", label="Warm start")
axes[0, 0].set_xlabel("Weeks of data (0..w)")
axes[0, 0].set_ylabel(metric_key)
axes[0, 0].set_title(f"Library ALS — {metric_key}")
axes[0, 0].legend()

# Library ALS: time
axes[0, 1].bar([w - 0.2 for w in weeks],
               [cold_times[w] for w in weeks], 0.4, label="Cold start")
axes[0, 1].bar([w + 0.2 for w in weeks],
               [warm_times[w] for w in weeks], 0.4, label="Warm start")
axes[0, 1].set_xlabel("Weeks of data")
axes[0, 1].set_ylabel("Time (s)")
axes[0, 1].set_title("Library ALS — training time")
axes[0, 1].legend()

# From-scratch iALS: metrics
axes[1, 0].plot(weeks, [cold_metrics_s[w][metric_key] for w in weeks],
                "o-", label="Cold start")
axes[1, 0].plot(weeks, [warm_metrics_s[w][metric_key] for w in weeks],
                "s--", label="Warm start")
axes[1, 0].set_xlabel("Weeks of data (0..w)")
axes[1, 0].set_ylabel(metric_key)
axes[1, 0].set_title(f"From-scratch iALS — {metric_key}")
axes[1, 0].legend()

# From-scratch iALS: time
axes[1, 1].bar([w - 0.2 for w in weeks],
               [cold_times_s[w] for w in weeks], 0.4, label="Cold start")
axes[1, 1].bar([w + 0.2 for w in weeks],
               [warm_times_s[w] for w in weeks], 0.4, label="Warm start")
axes[1, 1].set_xlabel("Weeks of data")
axes[1, 1].set_ylabel("Time (s)")
axes[1, 1].set_title("From-scratch iALS — training time")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 9. Target Comparison

We train iALS on every target (simple + composite) using the **full training data** (weeks 0–24) and evaluate on the validation set (week 25).

- **Library ALS**: `timespent`, `watch_ratio` and `composite` targets
- **From-scratch iALS**: all simple targets + composite targets

In [ ]:
results = {}

lib_targets = {
    "timespent (lib)": R_timespent,
    "watch_ratio (lib)": R_watch_ratio,
    "composite: engagement (lib)": target_matrices["composite: engagement"],
    "composite: action (lib)": target_matrices["composite: action"],
    "composite: full (lib)": target_matrices["composite: full"],
}
for name, R_t in tqdm(lib_targets.items(), desc="Library ALS targets"):
    m = AlternatingLeastSquares(
        factors=N_FACTORS, regularization=REG, alpha=ALPHA,
        iterations=N_ITERS, random_state=42,
    )
    m.fit(R_t, show_progress=False)
    results[name] = evaluate(
        m.user_factors, m.item_factors,
        R_train_any, val_user_items, k=K,
    )
    print(f"{name:30s} {results[name]}")

scratch_targets = {k: v for k, v in target_matrices.items()}
for name, R_t in tqdm(scratch_targets.items(), desc="iALS targets"):
    m = ImplicitALS(
        n_factors=N_FACTORS, n_iters=N_ITERS, reg=REG, alpha=ALPHA,
        random_state=42,
    )
    m.fit(R_t)
    results[f"{name} (iALS)"] = evaluate(
        m.user_factors, m.item_factors,
        R_train_any, val_user_items, k=K,
    )
    print(f"{name + ' (iALS)':30s} {results[name + ' (iALS)']}")

In [ ]:
import pandas as pd

rows = []
for name, m in results.items():
    rows.append({"Target": name, **m})
results_df = pd.DataFrame(rows).sort_values(f"NDCG@{K}", ascending=False)
results_df = results_df.reset_index(drop=True)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, max(6, 0.5 * len(results_df))))
y_pos = np.arange(len(results_df))
metric_cols = [f"HitRate@{K}", f"MRR@{K}", f"NDCG@{K}"]
width = 0.25

for j, col in enumerate(metric_cols):
    ax.barh(y_pos + j * width, results_df[col], width, label=col, alpha=0.8)

ax.set_yticks(y_pos + width)
ax.set_yticklabels(results_df["Target"])
ax.set_xlabel("Score")
ax.set_title(f"Target Comparison (K={K})")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Summary

In this seminar we:

1. **Loaded and explored** the VK-LSVD short-video dataset (`up0.001_ip0.001` subsample) with 7 feedback signals and weekly temporal structure.

2. **Derived and implemented iALS from scratch** following Hu, Koren & Volinsky (2008), with the efficient $V^\top V + \text{correction}$ trick for the confidence-weighted normal equations.

3. **Used the `implicit` library** for optimized ALS with its native `partial_fit_users` / `partial_fit_items` incremental training API.

4. **Compared cold-start vs warm-start** training as data grows week by week:
   - Warm start converges to comparable quality with significantly fewer iterations
   - Training time savings grow as the dataset gets larger

5. **Constructed and compared target signals**:
   - **Simple targets**: raw `timespent`, normalized `watch_ratio`, binary signals (`like`, `share`, `bookmark`, `click_on_author`, `open_comments`)
   - **Composite targets**: weighted combinations emphasizing engagement, deliberate actions, or the full signal mix
   - `watch_ratio` normalizes for video duration, providing a cleaner engagement signal than raw `timespent`
   - Composite targets can outperform individual signals by combining complementary information

### Key takeaways

- **Incremental training** is practical: warm-starting from previous factors reduces computation while maintaining quality.
- **Target engineering matters**: the choice and combination of feedback signals significantly affects recommendation quality.
- **watch_ratio** (timespent / duration) is a more principled engagement metric than raw watch time, as it accounts for varying video lengths.
- **Composite targets** allow trading off between different types of user intent (passive viewing vs. active engagement).